In [ ]:
# Install NLTK for preprocessing
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Step 1: Import Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Step 2: Load Data
true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

true_df['label'] = 1
fake_df['label'] = 0

data = pd.concat([true_df, fake_df])
data = data.sample(frac=1).reset_index(drop=True)

# Step 3: Preprocess Text Function
stop_words = set(stopwords.words('english'))

def clean_text(text):
    tokens = word_tokenize(text.lower())  # lowercase & tokenize
    tokens = [word for word in tokens if word.isalpha()]  # remove punctuation
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords
    return " ".join(tokens)

data['text'] = data['text'].apply(clean_text)

# Step 4: Split Dataset
X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Step 5: TF-IDF Vectorizer with better config
tfidf = TfidfVectorizer(max_df=0.9, min_df=5, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

# Step 6: Model Definitions
models = {
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=100, random_state=42),
    'Passive Aggressive': PassiveAggressiveClassifier(max_iter=1000, random_state=42)
}

# Step 7: Train, Evaluate, and Compare
results = {}

for name, model in models.items():
    print(f"\n Training: {name}")
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)
    print(f" Accuracy: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    results[name] = model

# Step 8: Prediction Function
def predict_with_all_models(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    print("\n Predictions:")
    for name, model in results.items():
        pred = model.predict(vec)[0]
        print(f"{name}: {'Real' if pred == 1 else 'Fake'}")

# Step 9: Test Inputs
test_samples = [
    "NASA confirms new planet with signs of water and life.",
    "Scientists confirm that bleach cures all diseases.",
    "A new health policy was introduced to reduce hospital costs.",
    "Aliens spotted talking to UN delegates.",
]

for sample in test_samples:
    print("\n=============================")
    print(f" Input: {sample}")
    predict_with_all_models(sample)
